# Modelos baseline

Entrena y evalúa los cuatro modelos base con evaluación multi-semilla.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # raiz del proyecto para importar src
import logging
logging.basicConfig(level=logging.INFO)

In [2]:
import pandas as pd
from src import data_loader as dl, preprocessing as pp, config
from src.models import get_baselines
from src.evaluation import multi_seed_evaluation
df = dl.load_ckd_dataset()
X_train, X_test, y_train, y_test = dl.split_data(df)
prep = pp.CKDPreprocessor()
X_train_p = prep.fit_transform(X_train); X_test_p = prep.transform(X_test)
X_res, y_res = pp.apply_smote(X_train_p, y_train)

INFO:src.data_loader:Dataset crudo cargado: 400 filas x 25 columnas


INFO:src.data_loader:Particion estratificada: train=320, test=80 (positivos train=0.62, test=0.62)


INFO:src.preprocessing:CKDPreprocessor ajustado: 28 features de salida


INFO:src.preprocessing:SMOTE aplicado: 320 -> 400 filas (positivos 0.62 -> 0.50)


## Evaluación multi-semilla (42-46)

In [3]:
rows = []
for name in get_baselines():
    factory = lambda seed, _n=name: get_baselines(seed)[_n]
    s = multi_seed_evaluation(factory, X_res, y_res, X_test_p, y_test)
    rows.append({'Modelo': name, 'AUC': s['auc_roc']['mean'], 'Sensibilidad': s['recall']['mean'],
                 'Especificidad': s['specificity']['mean'], 'F1': s['f1']['mean']})
pd.DataFrame(rows).round(3)

INFO:src.evaluation:Evaluacion multi-seed sobre 5 semillas completada


INFO:src.evaluation:Evaluacion multi-seed sobre 5 semillas completada


INFO:src.evaluation:Evaluacion multi-seed sobre 5 semillas completada


INFO:src.evaluation:Evaluacion multi-seed sobre 5 semillas completada


,Modelo,AUC,Sensibilidad,Especificidad,F1
0,Regresion Logistica,1.000,0.92,1.0,0.958
1,SVM (RBF),1.000,0.98,1.0,0.990
2,Random Forest,1.000,0.98,1.0,0.990
3,XGBoost (default),0.999,0.94,1.0,0.969
